# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Randaadad/FlyRank-AI/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Research Question

Can a Machine Learning model provide a useful directional signal for prioritizing pages that may represent CTR or engagement opportunities?

Decision Supported

The model is intended to help a human reviewer decide which pages should be reviewed first. The output is a prioritization signal rather than an automatic content decision.

In [ ]:
research_question = "Can a Machine Learning model help prioritize pages for CTR/engagement review?"
decision_supported = "Prioritize pages for human review"

print("Research question:", research_question)
print("Decision supported:", decision_supported)

Data

Dataset:
[Write the exact FlyRank dataset/release used.]

Tables or files:
[Write the exact tables/files used.]

Date window:
[Write the actual date range used.]

Unit of analysis:
One row represents [write your actual unit of analysis].

Exclusions:
I excluded [actual excluded observations/features] because [actual reason].

Only information necessary for the modeling task was retained in the public-facing analysis. Client names, private queries, and other private information were not included.

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Date/month used: March 2026")

Methodology

The task is framed as a page-level opportunity scoring problem.

Features

The model uses the final feature set defined during the modeling work. These features represent search and engagement signals available for the prediction task.

Label Definition

The target is a proxy for CTR/engagement opportunity constructed from the available search and engagement signals.

Baseline

A simple baseline was used as a reference point so that the Machine Learning model could be evaluated against a non-ML approach on the same validation split.

Validation Design

The model was evaluated using the validation design established during the validation audit. The purpose was to estimate performance on unseen data while reducing the risk of overly optimistic results.

Leakage Checks

The final feature set was reviewed for possible leakage. Features that could contain information unavailable at prediction time were excluded or treated carefully.

The model output is interpreted as a directional score for decision-support, not as a causal estimate of future CTR improvement.

In [ ]:
print("Final features:")
for feature in features:
    print("-", feature)
print("Target:", target_column)
print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))


Results

The Machine Learning model was compared with the baseline using the same validation split and evaluation metric.

The comparison is intended to show whether the model provides additional predictive signal compared with the simpler baseline.

The results are measured on the selected validation data and should not be interpreted as guaranteed performance on future data.

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

baseline_score = (
    test_df["expected_ctr"] - test_df["ctr"]
)

baseline_prediction = (
    baseline_score > 0
).astype(int)

model_prediction = (
    model_probability >= 0.5
).astype(int)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, baseline_score),
        roc_auc_score(y_test, model_probability)
    ],
    "Average_Precision": [
        average_precision_score(y_test, baseline_score),
        average_precision_score(y_test, model_probability)
    ],
    "Precision": [
        precision_score(
            y_test,
            baseline_prediction,
            zero_division=0
        ),
        precision_score(
            y_test,
            model_prediction,
            zero_division=0
        )
    ],
    "Recall": [
        recall_score(
            y_test,
            baseline_prediction,
            zero_division=0
        ),
        recall_score(
            y_test,
            model_prediction,
            zero_division=0
        )
    ]
})

display(comparison)

Limitations

This analysis has several limitations.

First, the target is a proxy for CTR or engagement opportunity rather than a direct measure of future business impact.

Second, the model identifies directional patterns but does not establish that changing a page will cause CTR to increase.

Third, the available features do not capture every factor that may influence search behavior or content performance.

Fourth, model performance depends on the dataset, feature set, target definition, and validation design.

The model should therefore be used as decision-support for prioritization rather than as an automatic content optimization system.

Human review is required before taking action based on the recommendations.

In [ ]:
import matplotlib.pyplot as plt

comparison.plot(
    x="method",
    y="ROC_AUC",
    kind="bar",
    legend=False
)

plt.title("ROC-AUC: Baseline vs Logistic Regression")
plt.ylabel("ROC-AUC")
plt.xlabel("")
plt.tight_layout()
plt.show()

comparison.plot(
    x="method",
    y="Average_Precision",
    kind="bar",
    legend=False
)

plt.title("Average Precision: Baseline vs Logistic Regression")
plt.ylabel("Average Precision")
plt.xlabel("")
plt.tight_layout()
plt.show()

Ranked Recommendations

The model output is converted into a ranked queue to help prioritize pages for human review.

Pages with higher opportunity scores receive higher review priority.

Each recommendation includes a reason code to make the ranking easier to interpret.

The recommended workflow is:

Opportunity score → Ranked queue → Human review → Content decision

The model does not automatically change, publish, delete, or rewrite content.

A human reviewer should consider search intent, content quality, existing performance, and business context before taking action.

The recommendations are therefore intended as directional decision-support rather than guaranteed optimization actions.

In [ ]:
action_queue.head(20)
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

action_queue.to_csv(
    output_dir / "ranked_action_queue.csv",
    index=False
)

print("Ranked action queue exported.")


Artifacts

The deployed paper will include the main evaluation results and visualizations generated from this notebook.

The main artifacts are:

1. Model versus baseline performance comparison.
2. Ranked action queue.
3. Supporting model or score distribution visualization.
4. Evaluation metrics used during validation.

These artifacts are generated from the same analysis used in the notebook so that the paper remains traceable to the underlying work.

In [ ]:
action_queue = test_df[
    [
        "content_hash_id",
        "report_date",
        "ctr",
        "expected_ctr",
        "average_position"
    ]
].copy()

action_queue["opportunity_score"] = model_probability

action_queue["reason_code"] = np.where(
    action_queue["ctr"] < action_queue["expected_ctr"],
    "CTR below position-tier expectation",
    "CTR not below position-tier expectation"
)

action_queue = action_queue.sort_values(
    "opportunity_score",
    ascending=False
)

display(action_queue.head(20))

from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

action_queue.to_csv(
    output_dir / "ranked_action_queue.csv",
    index=False
)

comparison.to_csv(
    output_dir / "model_vs_baseline.csv",
    index=False
)

print("Exports created:")
print("- work/outputs/ranked_action_queue.csv")
print("- work/outputs/model_vs_baseline.csv")

# ML-12 — Tell the Story

## 5-Minute Demo Outline

### 0:00–0:45 — The Question

The problem is that a content team may have many pages to review. I wanted to investigate whether Machine Learning could provide a useful directional signal for prioritizing pages that may represent CTR opportunities.

### 0:45–1:30 — The Data

I used the FlyRank internship warehouse and page-level search and engagement performance data from March 2026.

The main signals include impressions, clicks, CTR, average position, and available session-related features.

### 1:30–2:30 — The Method

I created a directional opportunity label by comparing each page's observed CTR with the typical CTR for its average-position tier.

I then trained a Logistic Regression model and compared it with a simpler Week-4 directional baseline.

The evaluation uses a client-grouped holdout so that the same client does not appear in both training and test data.

### 2:30–3:15 — The Result

The model and baseline were evaluated using the same held-out test data with ROC-AUC and Average Precision.

The results are interpreted as measured performance under this validation design rather than as evidence of future causal improvement.

### 3:15–4:15 — The Recommendation

The model can be used to create a ranked review queue.

Pages with stronger opportunity scores can be reviewed first. A human reviewer should then consider search intent, content quality, and other contextual information before deciding whether an action is appropriate.

### 4:15–5:00 — Limitations

The target is a proxy for CTR opportunity, not a direct future business outcome.

The model does not establish causality and should not automatically modify or publish content.

The final output is therefore best treated as directional decision-support for human review.


## Social Post

I built a Machine Learning workflow to investigate how pages could be prioritized for potential CTR opportunities using the FlyRank internship dataset.

I used page-level search and engagement signals to create a directional opportunity label, then compared a Logistic Regression model with a simpler rule-based baseline using client-grouped validation.

The final output is a ranked decision-support queue for human review rather than an automatic content optimization system.


## Employer-Facing Summary

I built a Logistic Regression-based opportunity scoring workflow using page-level search and engagement data from the FlyRank internship dataset. I compared the model with a simpler baseline and evaluated both using a client-grouped holdout to make the validation more realistic. The final output is a ranked decision-support queue that demonstrates my ability to turn Machine Learning results into practical, human-reviewed recommendations while clearly communicating limitations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
